In [3]:
from datetime import date
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd

# Dossier local du dépôt power-hpfc (courbes + settlements EEX de chaque séance)
HPFC_REPO = Path("/Users/clem/Documents/2-Coding/python/GitHub/power-hpfc")
MARKET = "FR"
PRICING_DATE = "latest"      # "latest" ou une date, ex. date(2026, 10, 13)
SYNC_GITHUB = True           # récupérer les dernières courbes du workflow avant de pricer

In [4]:
def git(repo: Path, *args: str) -> str:
    r = subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} a échoué dans {repo} :\n{r.stderr.strip()}")
    return r.stdout.strip()


if not (HPFC_REPO / "pyproject.toml").exists():
    raise FileNotFoundError(f"dépôt power-hpfc introuvable : {HPFC_REPO}")

if SYNC_GITHUB:
    if git(HPFC_REPO, "status", "--porcelain", "--", "data"):
        raise RuntimeError("des fichiers de data/ sont modifiés localement dans power-hpfc : "
                           "les annuler avant de synchroniser")
    git(HPFC_REPO, "fetch", "--quiet")
    behind = int(git(HPFC_REPO, "rev-list", "--count", "HEAD..@{u}"))
    if behind:
        git(HPFC_REPO, "pull", "--ff-only", "--quiet")
    print(f"power-hpfc : {behind} mise(s) à jour récupérée(s) depuis GitHub" if behind
          else "power-hpfc : déjà à jour avec GitHub")
else:
    print("Synchronisation désactivée : la courbe peut ne pas être la dernière")

# Le paquet hpfc importé doit être celui de ce dépôt (mêmes définitions que la courbe)
import hpfc
from hpfc import archive
from hpfc.config import load_config

if not Path(hpfc.__file__).resolve().is_relative_to(HPFC_REPO.resolve()):
    raise RuntimeError(f"le paquet hpfc importé vient de {hpfc.__file__}, pas de {HPFC_REPO} : "
                       "réinstaller avec `python -m pip install -e` depuis ce dépôt")
DATA = HPFC_REPO / "data"
cfg = load_config(MARKET)
print(f"Paquet hpfc : {Path(hpfc.__file__).parent}")

power-hpfc : déjà à jour avec GitHub
Paquet hpfc : /Users/clem/Documents/2-Coding/python/GitHub/power-hpfc/src/hpfc


In [5]:
available = archive.list_curves(MARKET, DATA)
if not available:
    raise FileNotFoundError(f"aucune courbe {MARKET} archivée dans {DATA}")
pricing_date = available[-1] if PRICING_DATE == "latest" else PRICING_DATE
if pricing_date not in available:
    near = [str(d) for d in available if abs((d - pricing_date).days) <= 7]
    raise FileNotFoundError(f"pas de courbe le {pricing_date} ; dates archivées proches : {near}")

curve = archive.load_curve(MARKET, pricing_date, DATA)
meta = archive.load_curve_metadata(MARKET, pricing_date, DATA)
settle = archive.load_settlements(MARKET, session=pricing_date, root=DATA)

if list(curve.columns) != ["local", "price", "is_peak"] or curve["price"].isna().any():
    raise ValueError("courbe au format inattendu ou incomplète")
if meta["curve"]["steps"] != len(curve):
    raise ValueError("la courbe ne correspond pas à ses métadonnées")

curve = curve.assign(local=pd.to_datetime(curve["local"]))
curve["date"] = curve["local"].dt.date
print(f"Courbe {MARKET} du {pricing_date} (méthode {meta['method']}, code {meta['code_version'][:7]}) : "
      f"{len(curve)} heures, du {curve['date'].iloc[0]} au {curve['date'].iloc[-1]}")
print(f"Settlements EEX de la séance : {len(settle)} produits cotés")
print(f"Courbes disponibles : {len(available)}, du {available[0]} au {available[-1]}")

Courbe FR du 2026-10-13 (méthode E, code 9a52e67) : 28201 heures, du 2026-10-14 au 2029-12-31
Settlements EEX de la séance : 64 produits cotés
Courbes disponibles : 37, du 2026-08-24 au 2026-10-13


In [6]:
def block_mean(curve: pd.DataFrame, start: date, end: date, profile: str) -> float:
    """Prix moyen de la courbe sur les jours [start, end], en base ou en peak."""
    m = (curve["date"] >= start) & (curve["date"] <= end)
    if profile == "Peak":
        m &= curve["is_peak"]
    if not m.any():
        raise ValueError(f"aucune heure {profile} entre le {start} et le {end}")
    return float(curve.loc[m, "price"].mean())


check = settle[["profile", "kind", "name", "start", "end", "settlement"]].copy()
check["courbe"] = [block_mean(curve, r.start, r.end, r.profile) for r in check.itertuples()]
check["ecart"] = check["courbe"] - check["settlement"]
worst = check["ecart"].abs().max()
if worst > cfg.arbitrage_tol:
    raise ValueError(f"la courbe ne reproduit pas les settlements (écart max {worst:.4f})")
print(f"Cohérence courbe / settlements : écart max {worst:.4f} {cfg.unit} sur {len(check)} produits")
check.round(2)

Cohérence courbe / settlements : écart max 0.0026 EUR/MWh sur 64 produits


,profile,kind,name,start,end,settlement,courbe,ecart
0,Base,day,D 14/10/26,2026-10-14,2026-10-14,191.12,191.12,0.0
1,Base,day,D 15/10/26,2026-10-15,2026-10-15,187.67,187.67,0.0
2,Base,day,D 16/10/26,2026-10-16,2026-10-16,185.95,185.95,0.0
3,Base,day,D 17/10/26,2026-10-17,2026-10-17,125.40,125.40,-0.0
4,Base,weekend,WE 17/10-18/10/26,2026-10-17,2026-10-18,119.47,119.47,-0.0
...,...,...,...,...,...,...,...,...
59,Peak,quarter,Q1-28,2028-01-01,2028-03-31,129.89,129.89,-0.0
60,Peak,year,Cal-28,2028-01-01,2028-12-31,76.48,76.48,0.0
61,Peak,quarter,Q2-28,2028-04-01,2028-06-30,28.99,28.99,0.0
62,Peak,quarter,Q3-28,2028-07-01,2028-09-30,39.98,39.98,0.0


2

In [11]:
from dataclasses import dataclass

# Grille horaire de la courbe : c'est l'unique grille sur laquelle un produit peut être défini
steps_s = curve.index.to_series().diff().dropna().dt.total_seconds()
if not (steps_s == 3600).all():
    raise ValueError("la courbe n'est pas horaire et continue : les volumes MW × h seraient faux")
DAY = curve["date"]
LOCAL_HOUR = curve["local"].dt.hour
WEEKDAY = curve["local"].dt.weekday          # 0 = lundi … 6 = dimanche
FIRST_DAY, LAST_DAY = DAY.iloc[0], DAY.iloc[-1]


@dataclass(frozen=True, eq=False)
class Product:
    """Produit linéaire : puissance signée (MW) heure par heure, sur la grille UTC de la courbe.
    Convention : MW > 0 = on achète l'électricité, MW < 0 = on la vend."""
    name: str
    mw: pd.Series

    def __post_init__(self):
        q = self.mw
        if q.empty:
            raise ValueError(f"{self.name} : aucune heure de livraison")
        if not q.index.is_unique or not q.index.isin(curve.index).all():
            raise ValueError(f"{self.name} : heures hors de la grille de la courbe "
                             f"({FIRST_DAY} → {LAST_DAY}) ou en double")
        if not np.isfinite(q.to_numpy(dtype=float)).all():
            raise ValueError(f"{self.name} : puissance manquante ou infinie")
        object.__setattr__(self, "mw", q.astype(float).sort_index())

    @property
    def volume(self) -> float:                       # MWh signés : pas horaire vérifié, 1 MW × 1 h = 1 MWh
        return float(self.mw.sum())

    def __add__(self, other: "Product") -> "Product":
        q = self.mw.add(other.mw, fill_value=0.0)
        return Product(f"{self.name} + {other.name}", q[q != 0])

    def __neg__(self) -> "Product":
        return Product(f"-({self.name})", -self.mw)

    def __sub__(self, other: "Product") -> "Product":
        q = self.mw.sub(other.mw, fill_value=0.0)
        return Product(f"{self.name} − {other.name}", q[q != 0])

    def __rmul__(self, k: float) -> "Product":
        return Product(f"{k:g} × {self.name}", k * self.mw)


print(f"Grille : {len(curve)} heures, du {FIRST_DAY} au {LAST_DAY} (heures locales {cfg.tz})")

Grille : 28201 heures, du 2026-10-14 au 2029-12-31 (heures locales Europe/Paris)


In [12]:
def _days(start: date, end: date) -> pd.Series:
    if start > end:
        raise ValueError(f"début {start} après la fin {end}")
    if start < FIRST_DAY or end > LAST_DAY:
        raise ValueError(f"période {start} → {end} hors de la courbe ({FIRST_DAY} → {LAST_DAY}) : "
                         "pas de prix pour ces heures")
    return (DAY >= start) & (DAY <= end)


def block(name: str, start: date, end: date, mw: float, profile: str = "Base") -> Product:
    """Bloc standard sur les jours [start, end] : Base, Peak (définition EEX) ou Offpeak (le reste)."""
    m = _days(start, end)
    if profile == "Peak":
        m &= curve["is_peak"]
    elif profile == "Offpeak":
        m &= ~curve["is_peak"]
    elif profile != "Base":
        raise ValueError(f"profil inconnu {profile!r} : Base, Peak ou Offpeak")
    return Product(name, pd.Series(float(mw), index=curve.index[m]))


def shaped(name: str, start: date, end: date, mw: float, hours, weekdays=range(7)) -> Product:
    """Bloc sur mesure : heures locales de début (0 = 00h-01h … 23 = 23h-24h) et jours de semaine (0 = lundi)."""
    hours, weekdays = set(hours), set(weekdays)
    if not hours <= set(range(24)) or not weekdays <= set(range(7)):
        raise ValueError("heures attendues dans 0…23, jours dans 0…6")
    m = _days(start, end) & LOCAL_HOUR.isin(hours) & WEEKDAY.isin(weekdays)
    return Product(name, pd.Series(float(mw), index=curve.index[m]))


def custom(name: str, mw: pd.Series) -> Product:
    """Profil libre : une puissance par heure, index horodaté avec fuseau (UTC ou local)."""
    if not isinstance(mw.index, pd.DatetimeIndex) or mw.index.tz is None:
        raise ValueError(f"{name} : l'index doit être horodaté avec un fuseau horaire")
    return Product(name, mw.tz_convert("UTC"))


def eex(name: str, profile: str = "Base", mw: float = 1.0) -> Product:
    """Produit EEX coté à la séance de la courbe (ex. 'Cal-27', 'Q1-27', 'M11-26'), reconstruit à l'identique."""
    row = settle[(settle["name"] == name) & (settle["profile"] == profile)]
    if len(row) != 1:
        quoted = sorted(settle.loc[settle["profile"] == profile, "name"])
        raise KeyError(f"{profile} {name} non coté le {pricing_date} ; cotés : {quoted}")
    r = row.iloc[0]
    return block(f"{profile} {name}", r["start"], r["end"], mw, profile)


def fair_price(p: Product) -> float:
    """Prix juste (EUR/MWh) : moyenne de la courbe pondérée par les volumes. Défini si tous les MW ont le même signe."""
    q = p.mw
    if (q > 0).any() and (q < 0).any():
        raise ValueError(f"{p.name} : achats et ventes mélangés, le prix par MWh n'a pas de sens "
                         "(utiliser la valeur en €, section 3)")
    return float((q * curve.loc[q.index, "price"]).sum() / q.sum())


def describe(*products: Product) -> pd.DataFrame:
    rows = []
    for p in products:
        local = curve.loc[p.mw.index, "local"]
        mixed = (p.mw > 0).any() and (p.mw < 0).any()
        rows.append({"produit": p.name, "début": local.iloc[0].date(), "fin": local.iloc[-1].date(),
                     "heures": len(p.mw), "MW min": p.mw.min(), "MW max": p.mw.max(),
                     "volume MWh": p.volume, "prix juste": np.nan if mixed else fair_price(p)})
    return pd.DataFrame(rows).set_index("produit")

print("Constructeurs prêts : block, shaped, custom, eex ; outils : fair_price, describe")

Constructeurs prêts : block, shaped, custom, eex ; outils : fair_price, describe


In [13]:
# Test de référence : un produit identique à un produit EEX doit redonner son settlement
cal = eex("Cal-27", "Base", mw=1)
cal_settl = settle.loc[(settle["name"] == "Cal-27") & (settle["profile"] == "Base"), "settlement"].item()
print(f"{cal.name} : {len(cal.mw)} heures, prix juste {fair_price(cal):.4f}, settlement {cal_settl:.2f}")
assert len(cal.mw) == 8760, "2027 doit compter 8760 heures"
assert abs(fair_price(cal) - cal_settl) <= cfg.arbitrage_tol

# Même test sur tous les produits cotés de la séance
rep = settle[["profile", "name", "settlement"]].copy()
rep["pricer"] = [fair_price(eex(r.name, r.profile)) for r in rep.itertuples()]
rep["ecart"] = rep["pricer"] - rep["settlement"]
assert rep["ecart"].abs().max() <= cfg.arbitrage_tol
print(f"{len(rep)} produits EEX repricés, écart max {rep['ecart'].abs().max():.4f} {cfg.unit}")

# Cohérence Peak + Offpeak = Base (heures et prix)
pk = block("Peak Cal-27", date(2027, 1, 1), date(2027, 12, 31), 1, "Peak")
op = block("Offpeak Cal-27", date(2027, 1, 1), date(2027, 12, 31), 1, "Offpeak")
assert len(pk.mw) + len(op.mw) == len(cal.mw) and len(pk.mw) == 261 * 12
recomposed = (fair_price(pk) * pk.volume + fair_price(op) * op.volume) / (pk.volume + op.volume)
assert abs(recomposed - fair_price(cal)) < 1e-9
print(f"Peak {len(pk.mw)} h + Offpeak {len(op.mw)} h = Base {len(cal.mw)} h ; prix recomposé {recomposed:.4f}")
print("Tous les tests passent")

Base Cal-27 : 8760 heures, prix juste 107.0926, settlement 107.09
64 produits EEX repricés, écart max 0.0026 EUR/MWh
Peak 3132 h + Offpeak 5628 h = Base 8760 h ; prix recomposé 107.0926
Tous les tests passent


In [14]:
# Exemples de produits structurés linéaires
ex_block = block("Achat 10 MW Base Q1-27", date(2027, 1, 1), date(2027, 3, 31), 10)
ex_shape = shaped("Achat 5 MW 6h-22h jours ouvrés Q1-27", date(2027, 1, 1), date(2027, 3, 31), 5,
                  hours=range(6, 22), weekdays=range(5))
ex_offpk = block("Vente 3 MW Offpeak M01-27", date(2027, 1, 1), date(2027, 1, 31), -3, "Offpeak")

# Profil libre : 8 MW la nuit et 2 MW le jour en janvier 2027 (à remplacer par un vrai profil heure par heure)
jan = curve.index[(DAY >= date(2027, 1, 1)) & (DAY <= date(2027, 1, 31))]
night = LOCAL_HOUR.loc[jan].isin(range(0, 6))
ex_custom = custom("Profil libre M01-27", pd.Series(np.where(night, 8.0, 2.0), index=jan))

ex_net = ex_block + ex_offpk                     # positions qui se compensent : 7 MW en offpeak de janvier
ex_swap = ex_block - block("Base Q2-27", date(2027, 4, 1), date(2027, 6, 30), 10)   # achat Q1, vente Q2
describe(ex_block, ex_shape, ex_offpk, ex_custom, ex_net, ex_swap).round(2)

,début,fin,heures,MW min,MW max,volume MWh,prix juste
produit,,,,,,,
Achat 10 MW Base Q1-27,2027-01-01,2027-03-31,2159,10.0,10.0,21590.0,168.06
Achat 5 MW 6h-22h jours ouvrés Q1-27,2027-01-01,2027-03-31,1024,5.0,5.0,5120.0,190.56
Vente 3 MW Offpeak M01-27,2027-01-01,2027-01-31,492,-3.0,-3.0,-1476.0,166.74
Profil libre M01-27,2027-01-01,2027-01-31,744,2.0,8.0,2604.0,171.16
Achat 10 MW Base Q1-27 + Vente 3 MW Offpeak M01-27,2027-01-01,2027-03-31,2159,7.0,10.0,20114.0,168.16
Achat 10 MW Base Q1-27 − Base Q2-27,2027-01-01,2027-06-30,4343,-10.0,10.0,-250.0,NaN


3

In [19]:
import io
import requests

# Courbe des taux zéro-coupon de la BCE (modèle de Svensson, obligations d'État de la zone euro)
ECB_CURVE = "G_N_A"          # G_N_A = émetteurs notés AAA ; G_N_C = tous les États de la zone euro
PAY_DAY = 20                 # paiement le 20 du mois suivant la livraison (convention EFET)
ECB_URL = "https://data-api.ecb.europa.eu/service/data/YC/"
PARAMS = ["BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
CHECKS = {"SR_1Y": 1.0, "SR_5Y": 5.0, "SR_10Y": 10.0}    # taux publiés, pour vérifier la formule


def ecb_yield_curve(day: date, curve_id: str = ECB_CURVE) -> dict[str, float]:
    keys = "+".join(PARAMS + list(CHECKS))
    r = requests.get(f"{ECB_URL}B.U2.EUR.4F.{curve_id}.SV_C_YM.{keys}",
                     params={"startPeriod": day.isoformat(), "endPeriod": day.isoformat(),
                             "format": "csvdata"}, timeout=30)
    if r.status_code == 404 or not r.text.strip():
        raise LookupError(f"pas de courbe BCE publiée le {day} (jour férié TARGET ?) : "
                          "aucun taux n'est substitué, choisir explicitement une autre date")
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    df = df[df["TIME_PERIOD"] == day.isoformat()]
    values = {k.split(".")[-1]: float(v) for k, v in zip(df["KEY"], df["OBS_VALUE"])}
    missing = [k for k in PARAMS + list(CHECKS) if k not in values or not np.isfinite(values[k])]
    if missing:
        raise LookupError(f"courbe BCE du {day} incomplète, séries manquantes : {missing}")
    return values


def svensson(p: dict[str, float], m) -> np.ndarray:
    """Taux zéro-coupon (en %, composition continue) pour des maturités m en années."""
    m = np.asarray(m, dtype=float)
    x1, x2 = m / p["TAU1"], m / p["TAU2"]
    f1 = (1 - np.exp(-x1)) / x1
    f2 = f1 - np.exp(-x1)
    f3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)
    return p["BETA0"] + p["BETA1"] * f1 + p["BETA2"] * f2 + p["BETA3"] * f3


ecb = ecb_yield_curve(pricing_date)
for k, m in CHECKS.items():                 # la formule doit redonner les taux publiés par la BCE
    if abs(float(svensson(ecb, m)) - ecb[k]) > 1e-3:
        raise ValueError(f"formule de Svensson incohérente avec {k} publié : "
                         f"{float(svensson(ecb, m)):.6f} contre {ecb[k]:.6f}")


def discount(pay) -> np.ndarray:
    """Facteurs d'actualisation à la date de pricing pour des dates de paiement, maturité en années ACT/365."""
    t = np.array([(d - pricing_date).days for d in pay], dtype=float) / 365.0
    if (t <= 0).any():
        raise ValueError("date de paiement antérieure ou égale à la date de pricing")
    return np.exp(-svensson(ecb, t) / 100 * t)


print(f"Courbe BCE {ECB_CURVE} du {pricing_date} : "
      + ", ".join(f"{k} {ecb[k]:.4f} %" for k in CHECKS) + " (formule vérifiée)")

Courbe BCE G_N_A du 2026-10-13 : SR_1Y 2.9250 %, SR_5Y 3.2070 %, SR_10Y 3.4918 % (formule vérifiée)


In [21]:
def cashflows(p: Product, K) -> pd.DataFrame:
    """Flux par mois de livraison : MWh, valeur au prix de la courbe, montant au prix K, actualisation."""
    k = pd.Series(float(K), index=p.mw.index) if np.isscalar(K) else K.tz_convert("UTC").reindex(p.mw.index)
    if k.isna().any():
        raise ValueError(f"{p.name} : prix K manquant sur {int(k.isna().sum())} heures")
    local = curve.loc[p.mw.index, "local"]
    df = pd.DataFrame({"mois": local.dt.strftime("%Y-%m"), "mwh": p.mw,
                       "courbe": p.mw * curve.loc[p.mw.index, "price"], "contrat": p.mw * k})
    m = df.groupby("mois")[["mwh", "courbe", "contrat"]].sum()
    year, month = m.index.str[:4].astype(int), m.index.str[5:].astype(int)
    m["paiement"] = [date(y + (mo == 12), mo % 12 + 1, PAY_DAY) for y, mo in zip(year, month)]
    m["df"] = discount(m["paiement"])
    m["valeur"] = m["courbe"] - m["contrat"]           # acheteur : reçoit l'électricité, paie K
    m["valeur act."] = m["valeur"] * m["df"]
    return m


def price(p: Product, K) -> dict:
    """Valeur du produit pour nous, face au prix contractuel K (EUR/MWh, constant ou heure par heure)."""
    m = cashflows(p, K)
    out = {"produit": p.name, "volume MWh": p.volume, "valeur €": m["valeur"].sum(),
           "valeur act. €": m["valeur act."].sum()}
    if not ((p.mw > 0).any() and (p.mw < 0).any()):
        out["prix juste"] = fair_price(p)                               # K qui annule la valeur non actualisée
        out["prix juste act."] = (m["courbe"] * m["df"]).sum() / (m["mwh"] * m["df"]).sum()
    return out

print(f"Valorisation prête : paiement le {PAY_DAY} du mois suivant la livraison, actualisation BCE {ECB_CURVE}")

Valorisation prête : paiement le 20 du mois suivant la livraison, actualisation BCE G_N_A


In [22]:
# Tests : au prix juste la valeur est nulle ; la valeur est linéaire ; le sens de la transaction change le signe
r = price(cal, cal_settl)
assert abs(price(cal, r["prix juste"])["valeur €"]) < 1e-6
assert abs(price(cal, r["prix juste act."])["valeur act. €"]) < 1e-6
a, b = price(ex_block, 100), price(ex_shape, 100)
assert abs(price(ex_block + ex_shape, 100)["valeur act. €"] - a["valeur act. €"] - b["valeur act. €"]) < 1e-6
assert abs(price(-ex_block, 100)["valeur act. €"] + a["valeur act. €"]) < 1e-6
assert abs(price(10 * cal, 100)["valeur act. €"] - 10 * price(cal, 100)["valeur act. €"]) < 1e-6
print(f"Achat 1 MW {cal.name} au settlement {cal_settl:.2f} : valeur {r['valeur €']:.2f} € "
      f"non actualisée (arrondi du settlement), {r['valeur act. €']:.2f} € actualisée")
print("Tous les tests passent")

Achat 1 MW Base Cal-27 au settlement 107.09 : valeur 22.66 € non actualisée (arrondi du settlement), 1380.48 € actualisée
Tous les tests passent


In [23]:
K = 100.0                    # prix contractuel, EUR/MWh
res = pd.DataFrame([price(p, K) for p in (ex_block, ex_shape, ex_offpk, ex_custom, ex_net, ex_swap)])
res.set_index("produit").round(2)

,volume MWh,valeur €,valeur act. €,prix juste,prix juste act.
produit,,,,,
Achat 10 MW Base Q1-27,21590.0,1469461.30,1453190.24,168.06,168.10
Achat 5 MW 6h-22h jours ouvrés Q1-27,5120.0,463688.94,458532.34,190.56,190.62
Vente 3 MW Offpeak M01-27,-1476.0,-98515.44,-97596.60,166.74,166.74
Profil libre M01-27,2604.0,185310.97,183582.59,171.16,171.16
Achat 10 MW Base Q1-27 + Vente 3 MW Offpeak M01-27,20114.0,1370945.86,1355593.65,168.16,168.20
Achat 10 MW Base Q1-27 − Base Q2-27,-250.0,2179261.30,2148956.19,NaN,NaN


4

In [24]:
def _hourly(p: Product, K) -> pd.DataFrame:
    k = pd.Series(float(K), index=p.mw.index) if np.isscalar(K) else K.tz_convert("UTC").reindex(p.mw.index)
    if k.isna().any():
        raise ValueError(f"{p.name} : prix K manquant sur {int(k.isna().sum())} heures")
    c = curve.loc[p.mw.index]
    return pd.DataFrame({"mois": c["local"].dt.strftime("%Y-%m"),
                         "bloc": np.where(c["is_peak"], "Peak", "Offpeak"),
                         "mwh": p.mw, "courbe": p.mw * c["price"], "contrat": p.mw * k})


def decompose(p: Product, K, by: str = "mois") -> pd.DataFrame:
    """Valeur du produit par période (mois, trimestre ou année) et par bloc Peak / Offpeak."""
    h = _hourly(p, K)
    h["df"] = h["mois"].map(cashflows(p, K)["df"])            # actualisation au mois de paiement
    year, month = h["mois"].str[:4], h["mois"].str[5:].astype(int)
    periods = {"mois": h["mois"], "trimestre": year + "-T" + ((month - 1) // 3 + 1).astype(str), "année": year}
    if by not in periods:
        raise ValueError(f"découpage inconnu {by!r} : {list(periods)}")
    h["periode"] = periods[by]
    h["act"] = (h["courbe"] - h["contrat"]) * h["df"]
    h["pos"], h["neg"] = h["mwh"] > 0, h["mwh"] < 0

    def table(g):
        t = g.agg(heures=("mwh", "size"), mwh=("mwh", "sum"), courbe=("courbe", "sum"),
                  contrat=("contrat", "sum"), act=("act", "sum"), pos=("pos", "any"), neg=("neg", "any"))
        return pd.DataFrame({
            "heures": t["heures"], "volume MWh": t["mwh"], "MW moyen": t["mwh"] / t["heures"],
            "prix courbe": (t["courbe"] / t["mwh"]).where(~(t["pos"] & t["neg"])),   # pas de prix si signes mixtes
            "valeur €": t["courbe"] - t["contrat"], "valeur act. €": t["act"]})

    out = table(h.groupby(["periode", "bloc"]))
    total = table(h.assign(periode="Total", bloc="").groupby(["periode", "bloc"]))
    return pd.concat([out, total])

print("Décomposition prête : par mois, trimestre ou année, et par bloc Peak / Offpeak")

Décomposition prête : par mois, trimestre ou année, et par bloc Peak / Offpeak


In [25]:
# Tests : la décomposition redonne la valeur totale, et les prix par bloc redonnent les settlements mensuels
for p in (cal, ex_shape, ex_net, ex_swap):
    for by in ("mois", "trimestre", "année"):
        d = decompose(p, 100, by)
        tot = price(p, 100)
        assert abs(d.loc[("Total", ""), "valeur act. €"] - tot["valeur act. €"]) < 1e-6
        assert abs(d.drop(index="Total")["valeur act. €"].sum() - tot["valeur act. €"]) < 1e-6
        assert d.drop(index="Total")["heures"].sum() == len(p.mw)

d = decompose(cal, 100, "mois")
months = settle[(settle["kind"] == "month") & (settle["start"] >= date(2027, 1, 1))
                & (settle["end"] <= date(2027, 12, 31))]
if months.empty:
    raise LookupError("aucun mois 2027 coté à cette séance : test impossible")
worst = 0.0
for r in months.itertuples():
    mois = r.start.strftime("%Y-%m")
    if r.profile == "Peak":
        px = d.loc[(mois, "Peak"), "prix courbe"]
    else:
        part = d.loc[mois]
        px = (part["prix courbe"] * part["volume MWh"]).sum() / part["volume MWh"].sum()
    worst = max(worst, abs(px - r.settlement))
assert worst <= cfg.arbitrage_tol
print(f"{len(months)} settlements mensuels 2027 retrouvés par la décomposition, écart max {worst:.4f} {cfg.unit}")
print("Tous les tests passent")

8 settlements mensuels 2027 retrouvés par la décomposition, écart max 0.0000 EUR/MWh
Tous les tests passent


In [26]:
decompose(ex_shape, K, "mois").round(2)

heures  volume MWh  MW moyen  prix courbe   valeur €  \
periode bloc                                                            
2027-01 Offpeak      84       420.0       5.0       191.15   38282.64   
        Peak        252      1260.0       5.0       224.42  156769.20   
2027-02 Offpeak      80       400.0       5.0       189.83   35933.23   
        Peak        240      1200.0       5.0       214.44  137328.00   
2027-03 Offpeak      92       460.0       5.0       193.37   42949.67   
        Peak        276      1380.0       5.0       137.99   52426.20   
Total              1024      5120.0       5.0       190.56  463688.94   

                 valeur act. €  
periode bloc                    
2027-01 Offpeak       37925.58  
        Peak         155307.03  
2027-02 Offpeak       35518.71  
        Peak         135743.79  
2027-03 Offpeak       42346.86  
        Peak          51690.38  
Total                458532.34

In [27]:
decompose(ex_swap, K, "trimestre").round(2)

heures  volume MWh  MW moyen  prix courbe   valeur €  \
periode bloc                                                            
2027-T1 Offpeak    1391     13910.0     10.00       155.82   776414.5   
        Peak        768      7680.0     10.00       190.24   693046.8   
2027-T2 Offpeak    1404    -14040.0    -10.00        73.23   375804.0   
        Peak        780     -7800.0    -10.00        57.18   333996.0   
Total              4343      -250.0     -0.06          NaN  2179261.3   

                 valeur act. €  
periode bloc                    
2027-T1 Offpeak      767707.86  
        Peak         685482.39  
2027-T2 Offpeak      368199.63  
        Peak         327566.31  
Total               2148956.19

5

In [28]:
from datetime import timedelta

# Produits de couverture : mois, trimestres et années cotés à la fois en Base et en Peak à la séance
def _month_end(y: int, m: int) -> date:
    return date(y + (m == 12), m % 12 + 1, 1) - timedelta(days=1)

std = settle[settle["kind"].isin(["month", "quarter", "year"])]
both = std.pivot_table(index=["start", "end", "name"], columns="profile", values="settlement", aggfunc="first")
both = both.dropna(subset=["Base", "Peak"]).reset_index()
QUOTED = {(r.start, r.end): (r.name, r.Base, r.Peak) for r in both.itertuples()}


def resolve(start: date, end: date, children) -> tuple[list, bool]:
    """Découpage le plus fin en produits cotés : les enfants s'ils couvrent tout, sinon le produit lui-même."""
    kids = [resolve(*c) for c in children]
    if kids and all(full for _, full in kids):
        return [x for pieces, _ in kids for x in pieces], True
    if (start, end) in QUOTED:
        return [(start, end)], True
    return [x for pieces, _ in kids for x in pieces], False


PIECES = []
for y in range(FIRST_DAY.year, LAST_DAY.year + 1):
    quarters = []
    for q in range(4):
        months = [(date(y, m, 1), _month_end(y, m), []) for m in range(3 * q + 1, 3 * q + 4)
                  if _month_end(y, m) >= FIRST_DAY and date(y, m, 1) <= LAST_DAY]
        if months:
            quarters.append((date(y, 3 * q + 1, 1), _month_end(y, 3 * q + 3), months))
    PIECES += resolve(date(y, 1, 1), date(y, 12, 31), quarters)[0]
PIECES.sort()

covered = pd.Series(False, index=curve.index)
for s, e in PIECES:
    covered |= (DAY >= s) & (DAY <= e)
UNCOVERED = sorted(set(DAY[~covered]))
print("Produits de couverture :", ", ".join(QUOTED[k][0] for k in PIECES))
print(f"Jours sans produit mois/trimestre/an coté : {len(UNCOVERED)}"
      + (f" ({UNCOVERED[0]} → {UNCOVERED[-1]})" if UNCOVERED else ""))

Produits de couverture : M11-26, M12-26, M01-27, M02-27, M03-27, Q2-27, Q3-27, Q4-27, Cal-28, Cal-29
Jours sans produit mois/trimestre/an coté : 18 (2026-10-14 → 2026-10-31)


In [29]:
def hedge(p: Product) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Couverture par bloc Base + Peak sur chaque produit coté : reproduit exactement les MWh Peak et Offpeak.
    Renvoie la table des couvertures, la position couverte (MW horaires) et le résidu de forme (MW horaires)."""
    q = p.mw.reindex(curve.index, fill_value=0.0)
    hedged = pd.Series(0.0, index=curve.index)
    rows = []
    for s, e in PIECES:
        m = (DAY >= s) & (DAY <= e)
        pk, op = m & curve["is_peak"], m & ~curve["is_peak"]
        v_pk, v_op = q[pk].sum(), q[op].sum()
        if v_pk == 0 and v_op == 0:
            continue
        base = v_op / op.sum()                        # MW Base : fixé par le volume offpeak
        peak = v_pk / pk.sum() - base                 # MW Peak en plus de la Base
        hedged[m] += base
        hedged[pk] += peak
        name, s_base, s_peak = QUOTED[(s, e)]
        rows.append({"produit EEX": name, "début": s, "fin": e,
                     "Base MW": base, "Peak MW": peak, "settl. Base": s_base, "settl. Peak": s_peak,
                     "MWh": base * m.sum() + peak * pk.sum(),
                     "montant settl. €": base * m.sum() * s_base + peak * pk.sum() * s_peak})
    table = pd.DataFrame(rows).set_index("produit EEX") if rows else pd.DataFrame()
    return table, hedged, q - hedged


def hedge_report(p: Product) -> dict:
    table, hedged, resid = hedge(p)
    px = curve["price"]
    out = {"produit": p.name, "volume MWh": p.volume,
           "valeur courbe €": float((p.mw * px[p.mw.index]).sum()),
           "couverture aux settlements €": float(table["montant settl. €"].sum()) if len(table) else 0.0,
           "forme résiduelle €": float((resid * px).sum()),
           "résidu |MWh|": float(resid.abs().sum()),
           "dont non couvrable MWh": float(p.mw[~covered[p.mw.index]].abs().sum())}
    out["arrondi settlements €"] = out["valeur courbe €"] - out["couverture aux settlements €"] - out["forme résiduelle €"]
    if not ((p.mw > 0).any() and (p.mw < 0).any()):
        out["prime de forme €/MWh"] = out["forme résiduelle €"] / p.volume
    return out

print("Couverture prête : Base + Peak sur le découpage le plus fin coté, résidu = risque de forme")

Couverture prête : Base + Peak sur le découpage le plus fin coté, résidu = risque de forme


In [30]:
# Tests : un produit EEX se couvre sans résidu ; chaque bloc de chaque produit coté est couvert au MWh près
for p in (cal, eex("Q1-27", "Peak", 3), eex("Cal-28")):
    r = hedge_report(p)
    assert r["résidu |MWh|"] < 1e-6 and abs(r["prime de forme €/MWh"]) < 1e-9, p.name
for p in (ex_shape, ex_custom, ex_swap):
    _, hedged, resid = hedge(p)
    for s, e in PIECES:
        m = (DAY >= s) & (DAY <= e)
        for blk in (m & curve["is_peak"], m & ~curve["is_peak"]):
            assert abs(resid[blk].sum()) < 1e-6, (p.name, s)
    r = hedge_report(p)
    assert abs(r["arrondi settlements €"]) <= cfg.arbitrage_tol * float(hedged.abs().sum()) + 1e-6
print("Tous les tests passent")

Tous les tests passent


In [31]:
hedge(ex_shape)[0].round(2)

,début,fin,Base MW,Peak MW,settl. Base,settl. Peak,MWh,montant settl. €
produit EEX,,,,,,,,
M01-27,2027-01-01,2027-01-31,0.85,4.15,186.28,224.42,1680.0,352802.05
M02-27,2027-02-01,2027-02-28,0.93,4.07,179.24,214.44,1600.0,321201.78
M03-27,2027-03-01,2027-03-31,0.99,4.01,139.71,137.99,1840.0,255160.40


In [32]:
rep5 = pd.DataFrame([hedge_report(p) for p in (ex_block, ex_shape, ex_offpk, ex_custom, ex_net, ex_swap)])
rep5.set_index("produit").round(2)

,volume MWh,valeur courbe €,couverture aux settlements €,forme résiduelle €,résidu |MWh|,dont non couvrable MWh,arrondi settlements €,prime de forme €/MWh
produit,,,,,,,,
Achat 10 MW Base Q1-27,21590.0,3628461.30,3628461.30,0.00,0.0,0.0,0.0,0.00
Achat 5 MW 6h-22h jours ouvrés Q1-27,5120.0,975688.94,929164.23,46524.71,2087.2,0.0,-0.0,9.09
Vente 3 MW Offpeak M01-27,-1476.0,-246115.44,-246115.44,0.00,0.0,0.0,-0.0,-0.00
Profil libre M01-27,2604.0,445710.97,463271.92,-17560.95,1388.2,0.0,0.0,-6.74
Achat 10 MW Base Q1-27 + Vente 3 MW Offpeak M01-27,20114.0,3382345.86,3382345.86,0.00,0.0,0.0,0.0,0.00
Achat 10 MW Base Q1-27 − Base Q2-27,-250.0,2154261.30,2154261.30,0.00,0.0,0.0,-0.0,NaN


6

In [33]:
TB_HOURS = (1, 2, 4)


def tb_select(prices: pd.Series, x: int) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Pour chaque jour local : les x heures les plus chères (top) et les x moins chères (bottom).
    Renvoie les masques top / bottom (index des prix) et le jour local de chaque heure."""
    day = pd.Series(prices.index.tz_convert(cfg.tz).date, index=prices.index)
    n = day.groupby(day).transform("size")
    if (n < 23).any() or (n > 25).any():
        raise ValueError("journées incomplètes dans la série de prix : TBx non calculable")
    if 2 * x > 23:
        raise ValueError(f"TB{x} impossible : 2 × {x} heures dépassent une journée")
    g = prices.groupby(day)
    top = g.rank(method="first", ascending=False) <= x
    bottom = g.rank(method="first", ascending=True) <= x
    return top, bottom, day


def tbx_daily(prices: pd.Series, x: int) -> pd.DataFrame:
    """Par jour : prix moyen des x heures les plus chères, des x moins chères, et leur écart TBx."""
    top, bottom, day = tb_select(prices, x)
    d = pd.DataFrame({"top": prices[top].groupby(day[top]).mean(),
                      "bottom": prices[bottom].groupby(day[bottom]).mean()})
    d[f"TB{x}"] = d["top"] - d["bottom"]
    return d


def tbx_mean(prices: pd.Series) -> pd.Series:
    """TBx moyen par jour (EUR/MWh) sur toute la série."""
    return pd.Series({f"TB{x}": tbx_daily(prices, x)[f"TB{x}"].mean() for x in TB_HOURS})


def tbx_monthly(prices: pd.Series) -> pd.DataFrame:
    """TBx moyen par jour (EUR/MWh), mois par mois."""
    cols = {}
    for x in TB_HOURS:
        d = tbx_daily(prices, x)[f"TB{x}"]
        cols[f"TB{x}"] = d.groupby([f"{t:%Y-%m}" for t in d.index]).mean()
    return pd.DataFrame(cols)


hpfc_px = curve["price"]
print(f"TBx prêts : TB{', TB'.join(map(str, TB_HOURS))} sur {DAY.nunique()} jours de courbe")

TBx prêts : TB1, TB2, TB4 sur 1175 jours de courbe


In [34]:
# TBx de la HPFC (valeur intrinsèque) par année, et TBx réalisés du spot sur les 12 derniers mois complets
spot_end = date(pricing_date.year, pricing_date.month, 1) - timedelta(days=1)
spot_start = date(spot_end.year - 1, spot_end.month + 1, 1) if spot_end.month < 12 else date(spot_end.year, 1, 1)
spot = archive.load_spot(MARKET, spot_start, spot_end, cfg.tz, hourly=True, root=DATA)

rows = {}
for y in range(FIRST_DAY.year, LAST_DAY.year + 1):
    m = (DAY >= max(date(y, 1, 1), FIRST_DAY)) & (DAY <= date(y, 12, 31))
    rows[f"HPFC {DAY[m].iloc[0]:%d/%m/%Y} → {DAY[m].iloc[-1]:%d/%m/%Y}"] = tbx_mean(hpfc_px[m])
rows[f"Spot réalisé {spot_start:%d/%m/%Y} → {spot_end:%d/%m/%Y}"] = tbx_mean(spot)
tb_summary = pd.DataFrame(rows).T
tb_summary.round(2)

,TB1,TB2,TB4
HPFC 14/10/2026 → 31/12/2026,121.27,113.82,99.22
HPFC 01/01/2027 → 31/12/2027,117.48,111.28,99.70
HPFC 01/01/2028 → 31/12/2028,76.28,71.78,63.84
HPFC 01/01/2029 → 31/12/2029,75.81,71.12,63.29
Spot réalisé 01/10/2025 → 30/09/2026,116.19,110.82,99.63


In [35]:
# Même mois calendaire : HPFC des 12 prochains mois contre spot réalisé des 12 derniers mois
next12 = hpfc_px[DAY < date(FIRST_DAY.year + 1, FIRST_DAY.month, 1)]
fwd_m, real_m = tbx_monthly(next12), tbx_monthly(spot)
cmp = pd.DataFrame(index=fwd_m.index)
cmp["mois spot"] = [f"{int(m[:4]) - 1}{m[4:]}" for m in fwd_m.index]
for x in TB_HOURS:
    cmp[f"HPFC TB{x}"] = fwd_m[f"TB{x}"]
    cmp[f"spot TB{x}"] = cmp["mois spot"].map(real_m[f"TB{x}"])
cmp.index.name = "mois HPFC"
cmp.round(2)

,mois spot,HPFC TB1,spot TB1,HPFC TB2,spot TB2,HPFC TB4,spot TB4
mois HPFC,,,,,,,
2026-10,2025-10,166.51,89.61,150.91,83.98,129.64,73.25
2026-11,2025-11,116.46,66.77,109.69,62.67,93.98,53.72
2026-12,2025-12,99.65,56.36,96.28,53.33,86.62,45.86
2027-01,2026-01,106.66,66.11,101.27,62.86,91.15,54.28
2027-02,2026-02,132.58,83.94,124.29,76.70,107.78,64.67
2027-03,2026-03,160.01,130.71,141.00,121.74,121.06,107.15
2027-04,2026-04,157.38,128.17,150.04,122.57,138.32,107.25
2027-05,2026-05,110.39,132.89,107.30,128.38,94.64,116.36
2027-06,2026-06,98.94,126.25,95.67,120.10,88.66,109.23


In [36]:
def battery(name: str, start: date, end: date, mw: float, x: int, eta: float) -> tuple[Product, pd.DataFrame]:
    """Batterie en valeur intrinsèque sur la HPFC : chaque jour, charge mw pendant les x heures les moins chères,
    restitue eta × mw pendant les x plus chères, seulement si eta × top > bottom. La chronologie (charger avant
    de décharger) et l'état de charge entre deux jours ne sont pas contraints : c'est la définition du TBx."""
    if not 0 < eta <= 1:
        raise ValueError("rendement aller-retour attendu dans ]0, 1]")
    px = hpfc_px[_days(start, end)]
    top, bottom, day = tb_select(px, x)
    d = tbx_daily(px, x)
    d["actif"] = eta * d["top"] > d["bottom"]
    on = day.map(d["actif"]).astype(bool)
    q = pd.Series(0.0, index=px.index)
    q[bottom & on] = mw                    # achat pour charger
    q[top & on] = -eta * mw                # vente de l'énergie restituée
    d["marge €"] = np.where(d["actif"], mw * x * (eta * d["top"] - d["bottom"]), 0.0)
    return Product(name, q[q != 0]), d


bat, bat_days = battery("Batterie 1 MW / 2 h, rendement 85 %, Cal-27", date(2027, 1, 1), date(2027, 12, 31),
                        mw=1, x=2, eta=0.85)
v = price(bat, 0.0)                         # K = 0 : la valeur est la marge d'arbitrage, au signe près
assert abs(-v["valeur €"] - bat_days["marge €"].sum()) < 1e-6
print(f"{bat.name} : {int(bat_days['actif'].sum())} jours actifs sur {len(bat_days)}")
print(f"Marge intrinsèque : {-v['valeur €']:,.0f} € non actualisée, {-v['valeur act. €']:,.0f} € actualisée")

Batterie 1 MW / 2 h, rendement 85 %, Cal-27 : 365 jours actifs sur 365
Marge intrinsèque : 62,477 € non actualisée, 61,068 € actualisée


7

In [37]:
# Backtest à périmètre identique : TBx de la HPFC publiée la veille contre TBx du spot réalisé le lendemain
bt_days = [d for d in available if d + timedelta(days=1) <= pricing_date]
if not bt_days:
    raise LookupError("aucune courbe archivée dont le lendemain est déjà réalisé")
first, last = bt_days[0] + timedelta(days=1), bt_days[-1] + timedelta(days=1)
spot_bt = archive.load_spot(MARKET, first, last, cfg.tz, hourly=True, root=DATA)
spot_day = spot_bt.index.tz_convert(cfg.tz).date

rows = []
for d in bt_days:
    nxt = d + timedelta(days=1)
    c = archive.load_curve(MARKET, d, DATA)["price"]
    hp = c[c.index.tz_convert(cfg.tz).date == nxt]
    sp = spot_bt[spot_day == nxt]
    if len(hp) != len(sp) or len(hp) < 23:
        raise ValueError(f"heures incohérentes le {nxt} : HPFC {len(hp)}, spot {len(sp)}")
    row = {"courbe du": d, "livraison": nxt, "HPFC base": hp.mean(), "spot base": sp.mean()}
    for x in TB_HOURS:
        row[f"HPFC TB{x}"] = tbx_daily(hp, x)[f"TB{x}"].item()
        row[f"spot TB{x}"] = tbx_daily(sp, x)[f"TB{x}"].item()
    rows.append(row)
bt = pd.DataFrame(rows).set_index("livraison")

summary = pd.DataFrame({
    "HPFC (moyenne)": [bt[f"HPFC TB{x}"].mean() for x in TB_HOURS],
    "spot (moyenne)": [bt[f"spot TB{x}"].mean() for x in TB_HOURS],
    "jours spot ≥ HPFC": [f"{(bt[f'spot TB{x}'] >= bt[f'HPFC TB{x}']).sum()} / {len(bt)}" for x in TB_HOURS],
}, index=[f"TB{x}" for x in TB_HOURS])
print(f"{len(bt)} jours livrés, du {first} au {last} ; base moyenne HPFC {bt['HPFC base'].mean():.2f}, "
      f"spot {bt['spot base'].mean():.2f} {cfg.unit}")
summary.round(2)

36 jours livrés, du 2026-08-25 au 2026-10-13 ; base moyenne HPFC 144.15, spot 144.15 EUR/MWh


,HPFC (moyenne),spot (moyenne),jours spot ≥ HPFC
TB1,165.71,181.29,20 / 36
TB2,151.73,171.80,23 / 36
TB4,134.90,156.76,24 / 36


7

In [38]:
import json
import re
from datetime import datetime, timezone

PRICER_REPO = HPFC_REPO.parent / "pricer"
if not (PRICER_REPO / ".git").exists():
    raise FileNotFoundError(f"dépôt pricer introuvable : {PRICER_REPO}")
PRICINGS = PRICER_REPO / "pricings"


def _commit(repo: Path) -> dict:
    r = subprocess.run(["git", "-C", str(repo), "rev-parse", "--verify", "--quiet", "HEAD"],
                       capture_output=True, text=True)
    return {"commit": r.stdout.strip() or None,            # None : dépôt sans aucun commit
            "modifications_locales": bool(git(repo, "status", "--porcelain"))}


def save_pricing(p: Product, K, label: str, overwrite: bool = False) -> Path:
    """Archive un pricing : position heure par heure, prix K, résultats, et tout ce qu'il faut pour le refaire."""
    slug = re.sub(r"[^A-Za-z0-9]+", "-", label).strip("-").lower()
    if not slug:
        raise ValueError("libellé vide")
    out = PRICINGS / f"{pricing_date}" / slug
    if out.exists() and not overwrite:
        raise FileExistsError(f"pricing déjà archivé : {out} (overwrite=True pour le remplacer)")
    out.mkdir(parents=True, exist_ok=True)

    k = pd.Series(float(K), index=p.mw.index) if np.isscalar(K) else K.tz_convert("UTC").reindex(p.mw.index)
    pd.DataFrame({"mw": p.mw, "k": k}).rename_axis("utc").to_parquet(out / "position.parquet")
    decompose(p, K, "mois").to_csv(out / "decomposition_mois.csv")
    table, _, _ = hedge(p)
    table.to_csv(out / "couverture.csv")

    res = {**price(p, K), **{f"couverture : {a}": b for a, b in hedge_report(p).items() if a != "produit"}}
    record = {
        "libelle": label, "produit": p.name, "archive_le_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "marche": MARKET, "date_pricing": str(pricing_date),
        "K": float(K) if np.isscalar(K) else "horaire (position.parquet, colonne k)",
        "courbe": {"methode": meta["method"], "code_version": meta["code_version"],
                   "heures": len(curve), "du": str(FIRST_DAY), "au": str(LAST_DAY)},
        "power_hpfc": _commit(HPFC_REPO), "pricer": _commit(PRICER_REPO),
        "actualisation": {"courbe_bce": ECB_CURVE, "parametres": ecb, "jour_paiement": PAY_DAY,
                          "decompte": "ACT/365", "composition": "continue"},
        "resultats": {a: (None if isinstance(b, float) and not np.isfinite(b) else b) for a, b in res.items()},
    }
    (out / "pricing.json").write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return out


def load_pricing(path: Path) -> tuple[dict, Product, pd.Series]:
    record = json.loads((path / "pricing.json").read_text(encoding="utf-8"))
    pos = pd.read_parquet(path / "position.parquet")
    return record, Product(record["produit"], pos["mw"]), pos["k"]

print(f"Archivage prêt : {PRICINGS}")

Archivage prêt : /Users/clem/Documents/2-Coding/python/GitHub/pricer/pricings


In [40]:
# Archive le profil 6h-22h au prix K, puis vérifie qu'on le relit et le reprice à l'identique
path = save_pricing(ex_shape, K, "Test profil 6h-22h Q1-27", overwrite=True)
saved, p2, k2 = load_pricing(path)
assert p2.mw.equals(ex_shape.mw)
again = price(p2, k2)
for key in ("valeur €", "valeur act. €", "prix juste", "prix juste act."):
    assert abs(again[key] - saved["resultats"][key]) < 1e-6, key
print(f"Archivé dans {path.relative_to(PRICER_REPO)} :", ", ".join(sorted(f.name for f in path.iterdir())))
print(f"Relu et repricé à l'identique : valeur actualisée {again['valeur act. €']:,.2f} €")
short = lambda c: c["commit"][:7] if c["commit"] else "aucun commit"
print(f"power-hpfc {short(saved['power_hpfc'])}, pricer {short(saved['pricer'])}"
      + (" (modifications non commitées)" if saved["pricer"]["modifications_locales"] else ""))

Archivé dans pricings/2026-10-13/test-profil-6h-22h-q1-27 : couverture.csv, decomposition_mois.csv, position.parquet, pricing.json
Relu et repricé à l'identique : valeur actualisée 458,532.34 €
power-hpfc 00211a4, pricer aucun commit (modifications non commitées)
